In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-ibm-catalog scikit-learn
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Singularity Machine Learning - Classification: A Qiskit Function by Multiverse Computing
> **Note:** * Qiskit Functions เป็นฟีเจอร์ทดลองที่ให้บริการเฉพาะผู้ใช้แผน IBM Quantum&reg; Premium Plan, Flex Plan และ On-Prem (ผ่าน IBM Quantum Platform API) เท่านั้น อยู่ในสถานะ preview release และอาจมีการเปลี่ยนแปลงได้

## ภาพรวม
ด้วยฟังก์ชัน "Singularity Machine Learning - Classification" คุณสามารถแก้ปัญหา machine learning ในโลกจริงบน quantum hardware ได้โดยไม่ต้องมีความเชี่ยวชาญด้าน quantum ฟังก์ชัน Application นี้อิงจาก ensemble methods และเป็น hybrid classifier ที่ใช้ประโยชน์จากวิธีแบบคลาสสิก เช่น boosting, bagging และ stacking สำหรับการฝึก ensemble เบื้องต้น จากนั้นจึงนำ quantum algorithms อย่าง variational quantum eigensolver (VQE) และ quantum approximate optimization algorithm (QAOA) มาใช้เพื่อเพิ่มความหลากหลาย ความสามารถในการ generalization และความซับซ้อนโดยรวมของ ensemble ที่ฝึกแล้ว

ต่างจาก quantum machine learning อื่น ๆ ฟังก์ชันนี้สามารถจัดการชุดข้อมูลขนาดใหญ่ที่มีตัวอย่างและ feature หลายล้านรายการได้ โดยไม่ถูกจำกัดด้วยจำนวน Qubit ใน QPU เป้าหมาย จำนวน Qubit จะกำหนดแค่ขนาดของ ensemble ที่สามารถฝึกได้เท่านั้น ฟังก์ชันนี้ยังมีความยืดหยุ่นสูง และสามารถใช้แก้ปัญหา classification ในหลากหลายโดเมน ทั้งด้านการเงิน สาธารณสุข และความปลอดภัยทางไซเบอร์
มันสามารถทำความแม่นยำสูงอย่างสม่ำเสมอบนปัญหาที่ท้าทายสำหรับคอมพิวเตอร์แบบคลาสสิก ซึ่งมีข้อมูลมิติสูง มีสัญญาณรบกวน และมีความไม่สมดุลของคลาส
![How it works](../docs/images/guides/multiverse-computing-singularity/how_it_works.avif)
เหมาะสำหรับ:
1. วิศวกรและนักวิทยาศาสตร์ข้อมูลในบริษัทที่ต้องการยกระดับผลิตภัณฑ์และบริการด้วยการนำ quantum machine learning มาผสานรวม
2. นักวิจัยในห้องปฏิบัติการวิจัย quantum ที่กำลังสำรวจการประยุกต์ใช้ quantum machine learning และต้องการใช้ quantum computing สำหรับงาน classification และ
3. นักศึกษาและอาจารย์ในสถาบันการศึกษาในรายวิชา machine learning ที่ต้องการแสดงให้เห็นถึงข้อได้เปรียบของ quantum computing

ตัวอย่างต่อไปนี้แสดงฟังก์ชันต่าง ๆ ของมัน ได้แก่ `create`, `list`, `fit` และ `predict` และแสดงวิธีใช้งานกับปัญหาสังเคราะห์ที่ประกอบด้วยครึ่งวงกลมสองอันเชื่อมต่อกัน ซึ่งเป็นปัญหาที่ท้าทายอย่างมากเนื่องจาก decision boundary ที่ไม่เป็นเส้นตรง


## คำอธิบายฟังก์ชัน
Qiskit Function นี้ช่วยให้ผู้ใช้สามารถแก้ปัญหา binary classification โดยใช้ quantum-enhanced ensemble classifier ของ Singularity เบื้องหลัง มันใช้แนวทาง hybrid ในการฝึก ensemble ของ classifier แบบคลาสสิกบนชุดข้อมูลที่มี label แล้วปรับแต่งให้มีความหลากหลายและ generalization สูงสุดโดยใช้ Quantum Approximate Optimization Algorithm (QAOA) บน IBM&reg; QPUs ผ่านอินเตอร์เฟซที่ใช้งานง่าย ผู้ใช้สามารถกำหนดค่า classifier ตามความต้องการ ฝึกมันบนชุดข้อมูลที่เลือก และใช้มันทำนายผลบนชุดข้อมูลที่ไม่เคยเห็นมาก่อน

เพื่อแก้ปัญหา classification ทั่วไป:
1. ประมวลผลชุดข้อมูลล่วงหน้า และแบ่งออกเป็นชุดฝึกและชุดทดสอบ ถ้าต้องการ สามารถแบ่งชุดฝึกออกเป็นชุดฝึกและชุด validation เพิ่มเติมได้ ทำได้โดยใช้ [scikit-learn](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html)
2. ถ้าชุดฝึกไม่สมดุล คุณสามารถทำการ resample เพื่อปรับสมดุลของคลาสได้โดยใช้ [imbalanced-learn](https://imbalanced-learn.org/stable/introduction.html#problem-statement-regarding-imbalanced-data-sets)
3. อัปโหลดชุดฝึก ชุด validation และชุดทดสอบแยกกันไปยัง storage ของฟังก์ชัน โดยใช้เมธอด `file_upload` ของ catalog และส่ง path ที่เกี่ยวข้องในแต่ละครั้ง
4. เริ่มต้น quantum classifier โดยใช้ action `create` ของฟังก์ชัน ซึ่งรับ hyperparameter ต่าง ๆ เช่น จำนวนและประเภทของ learner, regularization (ค่า lambda) และตัวเลือกการ optimization รวมถึงจำนวน layer, ประเภทของ classical optimizer, quantum Backend และอื่น ๆ
5. ฝึก quantum classifier บนชุดฝึกโดยใช้ action `fit` ของฟังก์ชัน โดยส่งชุดฝึกที่มี label และชุด validation (ถ้ามี)
6. ทำนายผลบนชุดทดสอบที่ไม่เคยเห็นมาก่อนโดยใช้ action `predict` ของฟังก์ชัน
## แนวทางแบบ action-based
ฟังก์ชันนี้ใช้แนวทางแบบ action-based ลองนึกภาพว่ามันเป็นสภาพแวดล้อมเสมือนที่คุณใช้ action เพื่อทำงานหรือเปลี่ยนสถานะของมัน ปัจจุบันมี action ดังต่อไปนี้: [list](#1-list), [create](#2-create), [delete](#3-delete), [fit](#4-fit), [predict](#5-predict), [fit_predict](#6-fit-predict) และ [create_fit_predict](#7-create-fit-predict) ตัวอย่างต่อไปนี้แสดงการใช้งาน action `create_fit_predict`

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load function
singularity = catalog.load("multiverse/singularity")

## Examples

### Classify a dataset

In this example, you'll use the "Singularity Machine Learning - Classification" function to classify a dataset consisting of two interleaving, moon-shaped half-circles. The dataset is synthetic, two-dimensional, and labeled with binary labels. It is created to be challenging for algorithms such as centroid-based clustering and linear classification.

![Moons dataset](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)

Through this process, you'll learn how to create the classifier, fit it to the training data, use it to predict on the test data, and delete the classifier when you're finished.

Before starting, you need to install [scikit-learn](https://scikit-learn.org/). Install it using the following command:

```sh
python3 -m pip install scikit-learn
```

Perform the following steps:

1) Create the synthetic dataset using the [`make_moons`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) function from [scikit-learn](https://scikit-learn.org/).
2) Upload the generated synthetic dataset to the [shared data directory](https://qiskit.github.io/qiskit-serverless/function_features/experimental/manage_data_directory.html).
3) Create the quantum-enhanced classifier using the [`create`](/docs/api/functions/multiverse-computing-singularity#create) action.
4) Enlist your classifiers using the [`list`](/docs/api/functions/multiverse-computing-singularity#list) action.
5) Train the classifier on the train data using the [`fit`](/docs/api/functions/multiverse-computing-singularity#fit) action.
6) Use the trained classifier to predict on the test data using the [`predict`](/docs/api/functions/multiverse-computing-singularity#predict) action.
7) Delete the classifier using the [`delete`](/docs/api/functions/multiverse-computing-singularity#delete) action.
8) Clean up after you're done.

**Step 1.** Import the necessary modules and generate the synthetic dataset, then split it into training and test datasets.

In [2]:
# import the necessary modules for this example
import os
import tarfile
import numpy as np

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# generate the synthetic dataset
X, y = make_moons(n_samples=10000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# print the first 10 samples of the training dataset
print("Features:", X_train[:10, :])
print("Targets:", y_train[:10])

Features: [[ 0.84757037 -0.48831433]
 [ 0.98132552  0.19235443]
 [-0.71626723  0.6978261 ]
 [ 1.18957848 -0.48186557]
 [ 0.52118982 -0.37791846]
 [ 0.81115408  0.58483251]
 [ 0.48706462  0.87336593]
 [-0.81880144  0.57407682]
 [ 1.67335408 -0.23932015]
 [ 0.50181306  0.8649761 ]]
Targets: [1 0 0 1 1 0 0 0 1 0]


### 1. List
action `list` จะดึงข้อมูล classifier ทั้งหมดที่จัดเก็บในรูปแบบ `*.pkl.tar` จาก shared data directory คุณสามารถเข้าถึงเนื้อหาของ directory นี้ได้โดยใช้เมธอด `catalog.files()` ด้วย โดยทั่วไป action list จะค้นหาไฟล์ที่มีนามสกุล `*.pkl.tar` ใน shared data directory และส่งคืนผลลัพธ์ในรูปแบบ list
#### Inputs

|     Name    |    Type     | Description |   Required  |
|-------------|-------------|-------------|-------------|
| `action` | `str` | ชื่อของ action จาก `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` และ `delete` | Yes |

#### Usage

In [3]:
def make_tarfile(file_path, tar_file_name):
    with tarfile.open(tar_file_name, "w") as tar:
        tar.add(file_path, arcname=os.path.basename(file_path))


# save the training and test datasets on your local disk
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

# create tar files for the datasets
make_tarfile("X_train.npy", "X_train.npy.tar")
make_tarfile("y_train.npy", "y_train.npy.tar")
make_tarfile("X_test.npy", "X_test.npy.tar")
make_tarfile("y_test.npy", "y_test.npy.tar")

# upload the datasets to the shared data directory
catalog.file_upload("X_train.npy.tar", singularity)
catalog.file_upload("y_train.npy.tar", singularity)
catalog.file_upload("X_test.npy.tar", singularity)
catalog.file_upload("y_test.npy.tar", singularity)

# view/enlist the uploaded files in the shared data directory
print(catalog.files(singularity))

['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar']


### 2. Create
action `create` จะสร้าง classifier ตามประเภท `quantum_classifier` ที่ระบุโดยใช้พารามิเตอร์ที่ให้มา และบันทึกไว้ใน shared data directory

> **Note:** ขณะนี้ฟังก์ชันรองรับเฉพาะ `QuantumEnhancedEnsembleClassifier` เท่านั้น
#### Inputs

|     Name    |    Type     | Description | Required  | Default |
|-------------|-------------|-------------|-----------|---------|
| `action` | `str` | ชื่อของ action จาก `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` และ `delete` | Yes | - |
| `name` | `str` | ชื่อของ quantum classifier เช่น `spam_classifier` | Yes | - |
| `instance` | `str` | IBM instance | Yes | - |
| `backend_name` | `str` | IBM compute resource ค่าเริ่มต้นคือ `None` ซึ่งหมายถึงจะใช้ Backend ที่มี pending jobs น้อยที่สุด | No | `None` |
| `quantum_classifier` | `str` | ประเภทของ quantum classifier คือ `QuantumEnhancedEnsembleClassifier` | No | `QuantumEnhancedEnsembleClassifier` |
| `num_learners` | `integer` | จำนวน learner ใน ensemble | No | `10` |
| `learners_types` | `list` | ประเภทของ learner ที่รองรับได้แก่: `DecisionTreeClassifier`, `GaussianNB`, `KNeighborsClassifier`, `MLPClassifier` และ `LogisticRegression` รายละเอียดเพิ่มเติมของแต่ละตัวสามารถดูได้ที่ [scikit-learn documentation](https://scikit-learn.org/stable/supervised_learning.html) | No | `[DecisionTreeClassifier]` |
| `learners_proportions` | `list` | สัดส่วนของ learner แต่ละประเภทใน ensemble | No | `[1.0]` |
| `learners_options` | `list` | ตัวเลือกสำหรับ learner แต่ละประเภทใน ensemble สำหรับรายการตัวเลือกทั้งหมดที่สอดคล้องกับประเภท learner ที่เลือก ให้ดู [scikit-learn documentation](https://scikit-learn.org/stable/supervised_learning.html) | No | `[{"max_depth": 3, "splitter": "random", "class_weight": None}]` |
| `regularization_type` | `str` or `list` | ประเภทของ regularization ที่จะใช้: `onsite` หรือ `alpha` `onsite` ควบคุม onsite term โดยค่าที่สูงกว่าจะทำให้ ensemble เบาบางมากขึ้น `alpha` ควบคุมการแลกเปลี่ยนระหว่าง interaction และ onsite term โดยค่าที่ต่ำกว่าจะทำให้ ensemble เบาบางมากขึ้น ถ้าระบุเป็น list จะฝึก model สำหรับแต่ละประเภทและเลือกตัวที่ดีที่สุด | No | `onsite` |
| `regularization` | `str` or `float` or `list` | ค่า regularization อยู่ระหว่าง `0` ถึง `+inf` ถ้า regularization_type เป็น `onsite` อยู่ระหว่าง `0` ถึง `1` ถ้า regularization_type เป็น `alpha` ถ้าตั้งเป็น `auto` จะใช้ auto-regularization — หา regularization parameter ที่เหมาะสมที่สุดด้วย binary search โดยใช้อัตราส่วนที่ต้องการของ classifier ที่เลือกต่อ classifier ทั้งหมด (`regularization_desired_ratio`) และขอบบนของ regularization parameter (`regularization_upper_bound`) ถ้าระบุเป็น list จะฝึก model สำหรับแต่ละค่าและเลือกตัวที่ดีที่สุด | No | `0.01` |
| `regularization_desired_ratio` | `float` or `list` | อัตราส่วนที่ต้องการของ classifier ที่เลือกต่อ classifier ทั้งหมดสำหรับ auto-regularization ถ้าระบุเป็น list จะฝึก model สำหรับแต่ละอัตราส่วนและเลือกตัวที่ดีที่สุด | No | `0.75` |
| `regularization_upper_bound` | `float` or `list` | ขอบบนของ regularization parameter เมื่อใช้ auto-regularization ถ้าระบุเป็น list จะฝึก model สำหรับแต่ละขอบบนและเลือกตัวที่ดีที่สุด | No | `200` |
| `weight_update_method` | `str` | วิธีอัปเดต sample weights จาก `logarithmic` และ `quadratic` | No | `logarithmic` |
| `sample_scaling` | `boolean` | ว่าจะใช้ sample scaling หรือไม่ | No | `False` |
| `prediction_scaling` | `float` | scaling factor สำหรับการทำนาย | No | `None` |
| `optimizer_options` | `dictionary` | ตัวเลือก QAOA optimizer รายการตัวเลือกทั้งหมดแสดงอยู่ในเอกสารนี้ด้านล่าง | No | ... |
| `voting` | `str` | ใช้ majority voting (`hard`) หรือค่าเฉลี่ยของ probability (`soft`) สำหรับรวบรวมการทำนาย/probability ของ learner | No | `hard` |
| `prob_threshold` | `float` | probability threshold ที่เหมาะสมที่สุด | No | `0.5` |
| `random_state` | `integer` | ควบคุมความสุ่มเพื่อให้ผลลัพธ์ซ้ำได้ | No | `None` |


- นอกจากนี้ `optimizer_options` มีรายการดังต่อไปนี้:

|     Name    |    Type     | Description | Required  | Default |
|-------------|-------------|-------------|-----------|---------|
| `num_solutions` | `integer` | จำนวน solution | No | `1024` |
| `reps` | `integer` | จำนวนรอบ | No | `4` |
| `sparsify` | `float` | sparsification threshold | No | `0.001` |
| `theta` | `float` | ค่าเริ่มต้นของ theta ซึ่งเป็น variational parameter ของ QAOA | No | `None` |
| `simulator` | `boolean` | ว่าจะใช้ simulator หรือ QPU | No | `False` |
| `classical_optimizer` | `str` | ชื่อของ classical optimizer สำหรับ QAOA solver ทั้งหมดที่ SciPy นำเสนอตามที่ระบุ[ที่นี่](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html#scipy.optimize.minimize) สามารถใช้ได้ คุณจะต้องตั้งค่า `classical_optimizer_options` ตามนั้น | No | `COBYLA` |
| `classical_optimizer_options` | `dictionary` | ตัวเลือก classical optimizer สำหรับรายการตัวเลือกทั้งหมด ดู [SciPy documentation](https://docs.scipy.org/doc/scipy/reference/) | No | `{"maxiter": 60}` |
| `optimization_level` | `integer` | ความลึกของ QAOA Circuit | No | `3` |
| `num_transpiler_runs` | `integer` | จำนวนรอบที่รัน Transpiler | No | `30` |
| `pass_manager_options` | `dictionary` | ตัวเลือกสำหรับการสร้าง preset pass manager | No | `{"approximation_degree": 1.0}` |
| `estimator_options` | `dictionary` | ตัวเลือก Estimator สำหรับรายการตัวเลือกทั้งหมด ดู [Qiskit Runtime Client documentation](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) | No | `None` |
| `sampler_options` | `dictionary` | ตัวเลือก Sampler สำหรับรายการตัวเลือกทั้งหมด ดู [Qiskit Runtime Client documentation](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options) | No | `None` |

- `estimator_options` เริ่มต้นคือ:

|     Name    |    Type     | Value  |
|-------------|-------------|-------------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `2` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |
| `resilience_options` | `dictionary` | `{"zne_mitigation": False, "zne": {"amplifier": "pea", "noise_factors": [1.0, 1.3, 1.6], "extrapolator": ["linear", "polynomial_degree_2", "exponential"],}}` |

- `sampler_options` เริ่มต้นคือ:

|     Name    |    Type     | Value |
|-------------|-------------|-------------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `1` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |

#### Usage

In [4]:
job = singularity.run(
    action="create",
    name="my_classifier",
    num_learners=10,
    learners_types=[
        "DecisionTreeClassifier",
        "KNeighborsClassifier",
    ],
    learners_proportions=[0.5, 0.5],
    learners_options=[{}, {}],
    regularization=0.01,
    weight_update_method="logarithmic",
    sample_scaling=True,
    optimizer_options={"simulator": True},
    voting="soft",
    prob_threshold=0.5,
)

print(job.result())

{'status': 'ok', 'message': 'Classifier created.', 'data': {}, 'metadata': {'resource_usage': {}}}


In [5]:
# list available classifiers using the list action
job = singularity.run(action="list")

print(job.result())

# you can also find your classifiers in the shared data directory with a *.pkl.tar extension
print(catalog.files(singularity))

{'status': 'ok', 'message': 'Classifiers listed.', 'data': {'classifiers': ['my_classifier']}, 'metadata': {'resource_usage': {}}}


['X_test.npy.tar', 'X_train.npy.tar', 'my_classifier.pkl.tar', 'y_test.npy.tar', 'y_train.npy.tar']


#### Validations

- `name`:
    - ชื่อต้องไม่ซ้ำกัน เป็น string ที่มีความยาวไม่เกิน 64 ตัวอักษร
    - สามารถมีได้เฉพาะตัวอักษรและตัวเลข (alphanumeric) และขีดล่างเท่านั้น
    - ต้องเริ่มต้นด้วยตัวอักษรและห้ามลงท้ายด้วยขีดล่าง
    - ต้องมี classifier ที่มีชื่อเดียวกันอยู่แล้วใน shared data directory
### 4. Fit
action `fit` ฝึก classifier โดยใช้ข้อมูลฝึกที่ให้มา

#### Inputs

|     Name    |    Type     | Description |   Required  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | ชื่อของ action ต้องเป็น `fit` | Yes |
| `name`      | `str`    | ชื่อของ classifier ที่จะฝึก | Yes |
| `X`         | `array` or `list` or `str` | ข้อมูลฝึก อาจเป็น NumPy array, list หรือ string ที่อ้างถึงชื่อไฟล์ใน shared data directory | Yes |
| `y`         | `array` or `list` or `str` | ค่า target สำหรับการฝึก อาจเป็น NumPy array, list หรือ string ที่อ้างถึงชื่อไฟล์ใน shared data directory | Yes |
| `fit_params`| `dictionary`| พารามิเตอร์เพิ่มเติมที่จะส่งให้เมธอด `fit` ของ classifier | No |

##### fit_params
|     Name    |    Type     | Description |   Required  |   Default   |
|-------------|-------------|-------------|-------------|-------------|
| `validation_data` | `tuple` | ข้อมูล validation และ label | No | `None` |
| `pos_label` | `integer` or `str` | label ของคลาสที่จะ map เป็น 1 | No | `None` |
| `optimization_data` | `str` | ชุดข้อมูลที่ใช้ optimize ensemble สามารถเป็น: `train`, `validation`, `both` | No | `train` |

#### Usage

In [6]:
job = singularity.run(
    action="fit",
    name="my_classifier",
    X="X_train.npy",  # you do not need to specify the tar extension
    y="y_train.npy",  # you do not need to specify the tar extension
)

print(job.result())

{'status': 'ok', 'message': 'Classifier fitted.', 'data': {}, 'metadata': {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 13.655871629714966}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 54.688621282577515}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 56.92286920547485}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 57.92738223075867}}}}


#### Validations

- `name`:
    - ชื่อต้องไม่ซ้ำกัน เป็น string ที่มีความยาวไม่เกิน 64 ตัวอักษร
    - สามารถมีได้เฉพาะตัวอักษรและตัวเลข (alphanumeric) และขีดล่างเท่านั้น
    - ต้องเริ่มต้นด้วยตัวอักษรและห้ามลงท้ายด้วยขีดล่าง
    - ต้องมี classifier ที่มีชื่อเดียวกันอยู่แล้วใน shared data directory
### 5. Predict
action `predict` ใช้เพื่อรับการทำนายแบบ hard และ soft (probabilities)

#### Inputs

|     Name    |    Type     | Description |   Required  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | ชื่อของ action ต้องเป็น `predict` | Yes |
| `name`      | `str`    | ชื่อของ classifier ที่จะใช้ | Yes |
| `X`         | `array` or `list` or `str` | ข้อมูลทดสอบ อาจเป็น NumPy array, list หรือ string ที่อ้างถึงชื่อไฟล์ใน shared data directory | Yes |
| `options["out"]` | `str` | ชื่อไฟล์ JSON สำหรับบันทึกการทำนายใน shared data directory ถ้าไม่ระบุ การทำนายจะถูกส่งคืนในผลลัพธ์ของ job | No |

#### Usage

In [7]:
job = singularity.run(
    action="predict",
    name="my_classifier",
    X="X_test.npy",  # you do not need to specify the tar extension
)

result = job.result()

print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results):", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results):", result["data"]["probabilities"][:5]
)

Action result status:  ok
Action result message:  Classifier predicted.
Predictions (first five results): [0, 0, 1, 0, 1]
Probabilities (first five results): [[1.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]]


#### Validations

- `name`:
    - ชื่อต้องไม่ซ้ำกัน เป็น string ที่มีความยาวไม่เกิน 64 ตัวอักษร
    - สามารถมีได้เฉพาะตัวอักษรและตัวเลข (alphanumeric) และขีดล่างเท่านั้น
    - ต้องเริ่มต้นด้วยตัวอักษรและห้ามลงท้ายด้วยขีดล่าง
    - ต้องมี classifier ที่มีชื่อเดียวกันอยู่แล้วใน shared data directory
- `options["out"]`:
    - ชื่อไฟล์ต้องไม่ซ้ำกัน เป็น string ที่มีความยาวไม่เกิน 64 ตัวอักษร
    - สามารถมีได้เฉพาะตัวอักษรและตัวเลข (alphanumeric) และขีดล่างเท่านั้น
    - ต้องเริ่มต้นด้วยตัวอักษรและห้ามลงท้ายด้วยขีดล่าง
    - ต้องมีนามสกุล `.json`
### 6. Fit-predict
action `fit_predict` ฝึก classifier โดยใช้ข้อมูลฝึก แล้วใช้มันเพื่อรับการทำนายแบบ hard และ soft (probabilities)

#### Inputs

|     Name    |    Type     | Description |   Required  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | ชื่อของ action ต้องเป็น `fit_predict` | Yes |
| `name`      | `str`    | ชื่อของ classifier ที่จะใช้ | Yes |
| `X_train`   | `array` or `list` or `str` | ข้อมูลฝึก อาจเป็น NumPy array, list หรือ string ที่อ้างถึงชื่อไฟล์ใน shared data directory | Yes |
| `y_train`   | `array` or `list` or `str` | ค่า target สำหรับการฝึก อาจเป็น NumPy array, list หรือ string ที่อ้างถึงชื่อไฟล์ใน shared data directory | Yes |
| `X_test`    | `array` or `list` or `str` | ข้อมูลทดสอบ อาจเป็น NumPy array, list หรือ string ที่อ้างถึงชื่อไฟล์ใน shared data directory | Yes |
| `fit_params`| `dictionary`| พารามิเตอร์เพิ่มเติมที่จะส่งให้เมธอด `fit` ของ classifier | No |
| `options["out"]` | `str` | ชื่อไฟล์ JSON สำหรับบันทึกการทำนายใน shared data directory ถ้าไม่ระบุ การทำนายจะถูกส่งคืนในผลลัพธ์ของ job | No |

#### Usage

In [8]:
job = singularity.run(
    action="delete",
    name="my_classifier",
)

# or you can delete from the shared data directory
# catalog.file_delete("my_classifier.pkl.tar", singularity)

print(job.result())

{'status': 'ok', 'message': 'Classifier deleted.', 'data': {}, 'metadata': {'resource_usage': {}}}


#### Validations

- `name`:
    - ชื่อต้องไม่ซ้ำกัน เป็น string ที่มีความยาวไม่เกิน 64 ตัวอักษร
    - สามารถมีได้เฉพาะตัวอักษรและตัวเลข (alphanumeric) และขีดล่างเท่านั้น
    - ต้องเริ่มต้นด้วยตัวอักษรและห้ามลงท้ายด้วยขีดล่าง
    - ต้องมี classifier ที่มีชื่อเดียวกันอยู่แล้วใน shared data directory

- `options["out"]`:
    - ชื่อไฟล์ต้องไม่ซ้ำกัน เป็น string ที่มีความยาวไม่เกิน 64 ตัวอักษร
    - สามารถมีได้เฉพาะตัวอักษรและตัวเลข (alphanumeric) และขีดล่างเท่านั้น
    - ต้องเริ่มต้นด้วยตัวอักษรและห้ามลงท้ายด้วยขีดล่าง
    - ต้องมีนามสกุล `.json`
### 7. Create-fit-predict
action `create_fit_predict` จะสร้าง classifier ฝึกมันโดยใช้ข้อมูลฝึกที่ให้มา แล้วใช้มันเพื่อรับการทำนายแบบ hard และ soft (probabilities)

#### Inputs

| Name | Type | Description | Required |
|------|------|-------------|-----------|
| `action` | `str` | ชื่อของ action จาก `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` และ `delete` | Yes |
| `name` | `str` | ชื่อของ classifier ที่จะใช้ | Yes |
| `quantum_classifier` | `str` | ประเภทของ classifier คือ `QuantumEnhancedEnsembleClassifier` ค่าเริ่มต้นคือ `QuantumEnhancedEnsembleClassifier` | No |
| `X_train` | `array` or `list` or `str` | ข้อมูลฝึก อาจเป็น NumPy array, list หรือ string ที่อ้างถึงชื่อไฟล์ใน shared data directory | Yes |
| `y_train` | `array` or `list` or `str` | ค่า target สำหรับการฝึก อาจเป็น NumPy array, list หรือ string ที่อ้างถึงชื่อไฟล์ใน shared data directory | Yes |
| `X_test` | `array` or `list` or `str` | ข้อมูลทดสอบ อาจเป็น NumPy array, list หรือ string ที่อ้างถึงชื่อไฟล์ใน shared data directory | Yes |
| `fit_params` | `dictionary` | พารามิเตอร์เพิ่มเติมที่จะส่งให้เมธอด `fit` ของ classifier | No |
| `options["save"]` | `boolean` | ว่าจะบันทึก trained classifier ใน shared data directory หรือไม่ ค่าเริ่มต้นคือ `True` | No |
| `options["out"]` | `str` | ชื่อไฟล์ JSON สำหรับบันทึกการทำนายใน shared data directory ถ้าไม่ระบุ การทำนายจะถูกส่งคืนในผลลัพธ์ของ job | No |

#### Usage

In [9]:
# delete the numpy files from your local disk
os.remove("X_train.npy")
os.remove("y_train.npy")
os.remove("X_test.npy")
os.remove("y_test.npy")

# delete the tar files from your local disk
os.remove("X_train.npy.tar")
os.remove("y_train.npy.tar")
os.remove("X_test.npy.tar")
os.remove("y_test.npy.tar")

# delete the tar files from the shared data
catalog.file_delete("X_train.npy.tar", singularity)
catalog.file_delete("y_train.npy.tar", singularity)
catalog.file_delete("X_test.npy.tar", singularity)
catalog.file_delete("y_test.npy.tar", singularity)

'Requested file was deleted.'

#### Validations

- `name`:
    - ถ้า `options["save"]` ตั้งเป็น `True`:
        - ชื่อต้องไม่ซ้ำกัน เป็น string ที่มีความยาวไม่เกิน 64 ตัวอักษร
        - สามารถมีได้เฉพาะตัวอักษรและตัวเลข (alphanumeric) และขีดล่างเท่านั้น
        - ต้องเริ่มต้นด้วยตัวอักษรและห้ามลงท้ายด้วยขีดล่าง
        - ไม่ควรมี classifier ที่มีชื่อเดียวกันอยู่แล้วใน shared data directory

- `options["out"]`:
    - ชื่อไฟล์ต้องไม่ซ้ำกัน เป็น string ที่มีความยาวไม่เกิน 64 ตัวอักษร
    - สามารถมีได้เฉพาะตัวอักษรและตัวเลข (alphanumeric) และขีดล่างเท่านั้น
    - ต้องเริ่มต้นด้วยตัวอักษรและห้ามลงท้ายด้วยขีดล่าง
    - ต้องมีนามสกุล `.json`
---
## Get started
ยืนยันตัวตนด้วย [IBM Quantum Platform API key](http://quantum.cloud.ibm.com/) แล้วเลือก Qiskit Function ดังนี้:

In [10]:
# Import QiskitFunctionsCatalog to load the
# "Singularity Machine Learning - Classification" function by Multiverse Computing
from qiskit_ibm_catalog import QiskitFunctionsCatalog

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# authentication
# If you have not previously saved your credentials, follow instructions at
# /docs/guides/functions
# to authenticate with your API key.
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load "Singularity Machine Learning - Classification" function by Multiverse Computing
singularity = catalog.load("multiverse/singularity")

# generate the synthetic dataset
X, y = make_moons(n_samples=1000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

job = singularity.run(
    action="create_fit_predict",
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    options={"save": False},
)

# get job status and result
status = job.status()
result = job.result()

print("Job status: ", status)
print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results): ", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results): ",
    result["data"]["probabilities"][:5],
)
print("Usage metadata: ", result["metadata"]["resource_usage"])

Job status:  QUEUED
Action result status:  ok
Action result message:  Classifier created, fitted, and predicted.
Predictions (first five results):  [0, 0, 1, 0, 0]
Probabilities (first five results):  [[0.87119766531518, 0.1288023346848197], [0.87119766531518, 0.1288023346848197], [0.24470328446479797, 0.7552967155352032], [0.820524432250189, 0.17947556774981072], [0.6847610293419495, 0.31523897065805173]]
Usage metadata:  {'RUNNING: MAPPING': {'CPU_TIME': 10.967791318893433}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 59.91712307929993}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 59.097386837005615}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 56.93338203430176}}


## Example
ในตัวอย่างนี้ เราจะใช้ฟังก์ชัน "Singularity Machine Learning - Classification" เพื่อจำแนกชุดข้อมูลที่ประกอบด้วยครึ่งวงกลมสองอันที่พันกันเป็นรูปพระจันทร์เสี้ยว ชุดข้อมูลนี้เป็นชุดสังเคราะห์ สองมิติ และมีป้ายกำกับแบบไบนารี ออกแบบมาให้ท้าทายกับอัลกอริทึม เช่น การจัดกลุ่มแบบ centroid-based และการจำแนกเชิงเส้น
![Moons dataset](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)
ในกระบวนการนี้ เราจะเรียนรู้วิธีสร้าง classifier, fit กับข้อมูล training, ใช้ทำนายข้อมูล test และลบ classifier เมื่อใช้งานเสร็จ
ก่อนเริ่มต้น ต้องติดตั้ง [scikit-learn](https://scikit-learn.org/) ก่อน ติดตั้งด้วยคำสั่งต่อไปนี้: